In [ ]:
"""
The recommended application pipeline is:
Step 1: Accept ticket input
Step 2: Validate ticket input
Step 3: Summarize the ticket
Step 4: Classify ticket category
Step 5: Analyze customer sentiment
Step 6: Detect priority and escalation risk
Step 7: Detect sensitive information
Step 8: Recommend internal routing
Step 9: Draft customer response
Step 10: Review generated response
Step 11: Build final ticket intelligence package
Step 12: Save output as JSON or Markdown
"""

In [ ]:
"""
Sentiment + priority + escalation risk
Classification + routing
Quality review + policy safety check
"""

In [ ]:
# prompt rules: Each prompt should contain:
"""
Each prompt should include:
1. Role
2. Task
3. Ticket input
4. Business rules
5. Constraints
6. Output schema
7. Hallucination control rule
"""

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise RuntimeError("GROQ_API_KEY is missing.")

GROQ_MODEL = os.getenv("GROQ_MODEL")
if not GROQ_MODEL:
    raise RuntimeError("GROQ_MODEL is missing.")

from groq import Groq
client = Groq(api_key=GROQ_API_KEY)


In [2]:
load_dotenv()

True

In [ ]:
#get user input for the blog post details
def get_user_input():
    print("Please provide the following information for ticket Generation:")
    
    customer_name = input("Customer Name: ".strip())
    customer_type = input("Customer Type (e.g., Free, paid, premium, enterprise): ".strip())
    ticket_subject = input("Ticket Subject: ".strip())
    ticket_body = input("Ticket Body: ".strip())
    product_area = input("Product Area (e.g., Billing, Technical Support, login): ".strip())
    previous_interaction_history = input("Previous Interaction History (if any): ".strip())
    sla_tier = input("SLA Tier (e.g., Standard, premium, enterprise): ".strip())
    response_tone = input("Response Tone (e.g., Professional, empathetic, concise, formal): ".strip())
    bussiness_rules = input("Business Rules (if any): ".strip())
    
customer_details = get_user_input()

def validate_customer_details(customer_details):
    ticket_subject = customer_details.get("ticket_subject")
    ticket_body = customer_details.get("ticket_body")
    response_tone = customer_details.get("response_tone")
    
    if not ticket_subject:
        raise ValueError("Ticket Subject is required.")
    if not ticket_body:
        raise ValueError("Ticket Body is required.")
    if not response_tone:
        raise ValueError("Response Tone is required.")
    return customer_details

validate_customer_details(customer_details)

In [ ]:
"""
Sentiment + priority + escalation risk
Classification + routing
Quality review + policy safety check
"""

In [ ]:
# Ticket Summarization
def call_groq_model(prompt:str, temperature:float=0.7, max_tokens:int=1500) -> str:
    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=prompt,
        temperature=temperature,
        max_tokens=max_tokens,
        response_format={"type": "json_object"}
    )
    return response.choices[0].message.content


In [5]:
# for testing purpose real input will be in JSON format and will be passed to the function
ticket_input="""
Subject: Charged twice and no response from support
Hi team,
I cancelled my premium subscription last month, but I was still 
charged again this month. I also noticed that the same invoice amount 
appears twice on my bank statement.
I contacted support two times last week but have not received any 
proper response. This is extremely frustrating. If this is not 
resolved today, I will escalate this publicly on LinkedIn and also ask
our finance team to block future payments.
Please refund the incorrect charge immediately.
Regards,
Amit

"""


# 1. Summarization

In [6]:
# Define the prompt template for ticket summarization

ticket_summarization_system_prompt = """
Role:
You are a senior support operations triage specialist expert in summarizing the ticket details.
Task:
Summarize the customer support ticket.
Think carefully about the category, customer impact, and escalation signals. Return only a concise reasoning summary, not detailed chain-of-thought

Rules:
- Do not invent account status, refund status, cancellation status, SLA commitments, legal conclusions, or policy promises.
- Do not include any information that is not explicitly stated in the ticket.
- Do not make assumptions about the customer's emotions, intentions, or future actions beyond what is explicitly stated.
- Return valid JSON only.

The summary should:
- Remove emotional noise while preserving urgency.
- Identify the core issue.
- Identify what the customer wants.
- Identify missing information needed for resolution.
- Not invent details.

Output JSON schema:
{
 "short_summary": "",
 "customer_problem": "",
 "business_impact": "",
 "customer_requested_action": "",
 "important_context": [],
 "missing_information": []
}
"""

ticket_summarization_user_prompt = """
Ticket:
{ticket_input}
summarize the above ticket according to the system prompt and return only the JSON output without any explanation or additional text as defined in the system prompt.
"""
summarization_prompt = [
    {"role": "system", "content": ticket_summarization_system_prompt},
    {"role": "user", "content": ticket_summarization_user_prompt.format(ticket_input=ticket_input)}
]


In [27]:
# take output in dict format parse it to json and save it to a file
ticket_summary=call_groq_model(summarization_prompt, temperature=0.7, max_tokens=500)
ticket_summary
# import json
# with open("ticket_summary.json", "w") as f:
#     json.dump(ticket_summary_dict, f, indent=4) 

# response = client.chat.completions.create(
#         model=GROQ_MODEL,
#         messages=summarization_prompt,
#         temperature=0.7,
#         max_tokens=500
#     )
# response.choices[0].message.content

'```\n{\n "short_summary": "Customer charged twice after cancelling premium subscription",\n "customer_problem": "Incorrect double charge after subscription cancellation",\n "business_impact": "Potential public escalation on LinkedIn and blocked future payments",\n "customer_requested_action": "Immediate refund of incorrect charge",\n "important_context": ["premium subscription cancelled last month", "double charge on bank statement", "previous support requests unanswered"],\n "missing_information": ["subscription cancellation confirmation", "invoice details", "previous support ticket references"]\n}\n```'

In [29]:
# Joutput SON parsing
import json

def parse_json_response(json_string:str, file_name:str):
    try:
        # save the output to json not string
        # if isinstance(json_string, str):
            # json_string = json.loads(json_string)
        with open(f"./outputs/{file_name}", "w") as f:
            json.dump(json_string, f, indent=4)
            print(f"Output saved to ./outputs/{file_name}")
    
    except json.JSONDecodeError as e:
        raise ValueError(f"Failed to parse JSON: {e}")
    
parse_json_response(ticket_summary, "sample_ticket_summary.json")

Output saved to ./outputs/sample_ticket_summary.json


In [ ]:
"""
Sentiment + priority + escalation risk
Classification + routing
Quality review + policy safety check
"""

# 2. classification

In [34]:
ticket_classification_system_prompt = """
Role:
You are a senior support operations triage specialist.

Task:
Analyze a customer support ticket and classify it based only on the information explicitly stated in the ticket.

Instructions:
- Do not invent information.
- Do not assume account status, refund status, cancellation status, SLA commitments, legal conclusions, or policy outcomes.
- Do not infer facts that are not explicitly mentioned.
- Base all classifications only on the ticket content.
- Keep reasoning summaries brief and factual.

Output Requirements:
- Return ONLY valid JSON.
- Do NOT return markdown.
- Do NOT return code fences.
- Do NOT return explanations.
- Do NOT return notes.
- Do NOT return any text before or after the JSON.
- The response must begin with '{' and end with '}'.

Output Schema:
{
  "classification": {
    "primary_category": "",
    "secondary_categories": [],
    "category_reasoning_summary": "",
    "confidence_score": 0.0
  },
  "sentiment_analysis": {
    "sentiment": "",
    "emotion_signals": [],
    "sentiment_reasoning_summary": "",
    "confidence_score": 0.0
  },
  "priority_and_risk": {
    "priority": "",
    "escalation_risk": "",
    "risk_triggers": [],
    "recommended_sla_action": "",
    "reasoning_summary": ""
  }
}

Example Output:
{
  "classification": {
    "primary_category": "BILLING_ISSUE",
    "secondary_categories": [
      "CANCELLATION_OR_REFUND",
      "ESCALATION_COMPLAINT"
    ],
    "category_reasoning_summary": "The ticket concerns billing after cancellation and mentions duplicate charges.",
    "confidence_score": 0.93
  },
  "sentiment_analysis": {
    "sentiment": "FRUSTRATED",
    "emotion_signals": [
      "Customer describes the experience as frustrating.",
      "Customer mentions multiple unresolved support contacts."
    ],
    "sentiment_reasoning_summary": "The customer expresses dissatisfaction and urgency.",
    "confidence_score": 0.95
  },
  "priority_and_risk": {
    "priority": "URGENT",
    "escalation_risk": "HIGH",
    "risk_triggers": [
      "Payment-related issue",
      "Repeated unresolved support attempts",
      "Threat of public escalation"
    ],
    "recommended_sla_action": "Route to billing support immediately.",
    "reasoning_summary": "Financial impact and escalation signals justify urgent handling."
  }
}
"""

ticket_classification_user_prompt = """
Customer Support Ticket:

{ticket_input}

Analyze the ticket and return a JSON response that exactly follows the schema defined in the system prompt.
"""

In [35]:
classification_prompt = [
    {
        "role": "system",
        "content": ticket_classification_system_prompt
    },
    {
        "role": "user",
        "content": ticket_classification_user_prompt.format(
            ticket_input=ticket_input
        )
    }
]

In [36]:
call_groq_model(classification_prompt, temperature=0.7, max_tokens=500)
# ticket_classification

'{\n  "classification": {\n    "primary_category": "BILLING_ISSUE",\n    "secondary_categories": [\n      "CANCELLATION_OR_REFUND",\n      "ESCALATION_COMPLAINT"\n    ],\n    "category_reasoning_summary": "The ticket concerns billing after cancellation and mentions duplicate charges.",\n    "confidence_score": 0.93\n  },\n  "sentiment_analysis": {\n    "sentiment": "FRUSTRATED",\n    "emotion_signals": [\n      "Customer describes the experience as frustrating.",\n      "Customer mentions multiple unresolved support contacts.",\n      "Customer threatens public escalation."\n    ],\n    "sentiment_reasoning_summary": "The customer expresses strong dissatisfaction and urgency.",\n    "confidence_score": 0.95\n  },\n  "priority_and_risk": {\n    "priority": "URGENT",\n    "escalation_risk": "HIGH",\n    "risk_triggers": [\n      "Payment-related issue",\n      "Repeated unresolved support attempts",\n      "Threat of public escalation"\n    ],\n    "recommended_sla_action": "Route to bil

In [ ]:
"""
Sentiment + priority + escalation risk
Classification + routing
Quality review + policy safety check
"""

In [ ]:
"""
Step 7: Detect sensitive information
Step 8: Recommend internal routing
Step 9: Draft customer response
Step 10: Review generated response
Step 11: Build final ticket intelligence package
Step 12: Save output as JSON or Markdown

"""

### 7. Detect sensitive information

In [37]:
# Sensitive Information Detection Prompt

sensitive_information_system_prompt = """
Role:
You are a data privacy and customer support compliance specialist.

Task:
Analyze the customer support ticket and determine whether it contains sensitive information or references to sensitive information.

Instructions:
- Analyze only the information explicitly stated in the ticket.
- Do not invent or assume the presence of sensitive information.
- Do not infer personal data that is not mentioned.
- Do not generate or reveal any sensitive information.
- If no sensitive information is detected, return an empty list for sensitive_categories and handling_recommendations.
- Keep evidence summaries concise and factual.

Possible Sensitive Categories:
- PAYMENT_INFORMATION
- PERSONAL_IDENTIFIABLE_INFORMATION
- ACCOUNT_CREDENTIALS
- FINANCIAL_INFORMATION
- GOVERNMENT_IDENTIFIERS
- HEALTH_INFORMATION
- CONTACT_INFORMATION
- SECURITY_INFORMATION
- OTHER_SENSITIVE_INFORMATION

Output Requirements:
- Return ONLY valid JSON.
- Do NOT return markdown.
- Do NOT return code fences.
- Do NOT return explanations.
- Do NOT return notes.
- Do NOT return any text before or after the JSON.
- The response must begin with '{' and end with '}'.

Output Schema:
{
  "sensitive_information_check": {
    "sensitive_information_detected": true,
    "sensitive_categories": [],
    "evidence_summary": "",
    "handling_recommendations": []
  }
}

Example Output:
{
  "sensitive_information_check": {
    "sensitive_information_detected": true,
    "sensitive_categories": [
      "PAYMENT_INFORMATION"
    ],
    "evidence_summary": "The customer references bank statement charges and subscription billing activity.",
    "handling_recommendations": [
      "Do not expose billing details unnecessarily.",
      "Request account verification through a secure channel.",
      "Escalate to billing support if account review is required."
    ]
  }
}
"""

sensitive_information_user_prompt = """
Customer Support Ticket:

{ticket_input}

Analyze the ticket and return a JSON response that exactly follows the schema defined in the system prompt.
"""

sensitive_information_prompt = [
    {
        "role": "system",
        "content": sensitive_information_system_prompt
    },
    {
        "role": "user",
        "content": sensitive_information_user_prompt.format(
            ticket_input=ticket_input
        )
    }
]




In [39]:
# Model Call

response = call_groq_model(
    sensitive_information_prompt,
    temperature=0,
    max_tokens=300,  
)

print(response)

{
  "sensitive_information_check": {
    "sensitive_information_detected": true,
    "sensitive_categories": [
      "PAYMENT_INFORMATION",
      "FINANCIAL_INFORMATION"
    ],
    "evidence_summary": "The customer mentions being charged twice, invoice amount on bank statement, and potential future payment blockage.",
    "handling_recommendations": [
      "Verify the customer's account and subscription status.",
      "Investigate the duplicate charge and process a refund if necessary.",
      "Respond to the customer in a timely manner to prevent public escalation."
    ]
  }
}


### Recommend Internal Routing

In [41]:
# Ticket Routing Recommendation Prompt

routing_recommendation_system_prompt = """
Role:
You are a senior customer support operations and ticket routing specialist.

Task:
Analyze the customer support ticket and determine the most appropriate team to handle the ticket.

Instructions:
- Base your analysis only on information explicitly stated in the ticket.
- Do not invent account status, refund eligibility, cancellation status, payment verification results, or policy outcomes.
- Do not make promises on behalf of support teams.
- Do not assume facts that are not mentioned in the ticket.
- Generate concise and factual routing recommendations.
- Follow-up information should only include information that would reasonably help the assigned team investigate the issue.
- If the ticket lacks sufficient information, include the most relevant missing details in required_follow_up_information.

Possible Teams:
- BILLING_SUPPORT
- TECHNICAL_SUPPORT
- ACCOUNT_SUPPORT
- CUSTOMER_SUCCESS
- SALES_SUPPORT
- SECURITY_TEAM
- COMPLIANCE_TEAM
- PRODUCT_SUPPORT
- GENERAL_SUPPORT
- ESCALATION_TEAM

Output Requirements:
- Return ONLY valid JSON.
- Do NOT return markdown.
- Do NOT return code fences.
- Do NOT return explanations.
- Do NOT return notes.
- Do NOT return any text before or after the JSON.
- The response must begin with '{' and end with '}'.

Output Schema:
{
  "routing_recommendation": {
    "recommended_team": "",
    "routing_reason": "",
    "internal_note": "",
    "required_follow_up_information": []
  }
}

Example Output:
{
  "routing_recommendation": {
    "recommended_team": "BILLING_SUPPORT",
    "routing_reason": "The ticket concerns cancellation, duplicate billing, refund inquiry, and payment-related questions.",
    "internal_note": "Review cancellation records, billing history, and transaction activity before making any refund determination.",
    "required_follow_up_information": [
      "Invoice ID",
      "Registered account email",
      "Cancellation confirmation",
      "Transaction date"
    ]
  }
}
"""

routing_recommendation_user_prompt = """
Customer Support Ticket:

{ticket_input}

Analyze the ticket and return a JSON response that exactly follows the schema defined in the system prompt.
"""

routing_recommendation_prompt = [
    {
        "role": "system",
        "content": routing_recommendation_system_prompt
    },
    {
        "role": "user",
        "content": routing_recommendation_user_prompt.format(
            ticket_input=ticket_input
        )
    }
]

# Model Call

response = call_groq_model(
    routing_recommendation_prompt,
    temperature=0,
    max_tokens=300,
)

print(response)

{
  "routing_recommendation": {
    "recommended_team": "BILLING_SUPPORT",
    "routing_reason": "The ticket concerns duplicate billing, cancellation, and refund inquiry.",
    "internal_note": "Review cancellation records, billing history, and transaction activity before making any refund determination.",
    "required_follow_up_information": [
      "Invoice ID",
      "Registered account email",
      "Cancellation confirmation",
      "Transaction date"
    ]
  }
}


### Genearte Response

In [42]:
# Draft Customer Response Generation Prompt

draft_response_system_prompt = """
Role:
You are a senior customer support response specialist.

Task:
Generate a professional customer support response based only on the information explicitly provided in the support ticket.

Instructions:
- Base the response only on the ticket content.
- Do not invent facts, account status, refund status, cancellation status, investigation results, policy decisions, or future actions.
- Do not promise refunds, compensation, credits, resolutions, or timelines unless explicitly stated in the ticket.
- Acknowledge customer concerns when appropriate.
- Maintain a professional, empathetic, and concise tone.
- If additional information is required, clearly request it.
- Include any assumptions that prevent a final resolution.
- List any information that must be obtained before the response can be safely sent.
- Keep response_strategy concise and action-oriented.

Output Requirements:
- Return ONLY valid JSON.
- Do NOT return markdown.
- Do NOT return code fences.
- Do NOT return explanations.
- Do NOT return notes.
- Do NOT return any text before or after the JSON.
- The response must begin with '{' and end with '}'.

Output Schema:
{
  "draft_customer_response": {
    "draft_response": "",
    "response_strategy": "",
    "assumptions": [],
    "information_needed_before_sending": []
  }
}

Example Output:
{
  "draft_customer_response": {
    "draft_response": "Hi Amit, thank you for contacting us. I understand your concern regarding the charge that appeared after your subscription cancellation. We would like to investigate this further. To help us review the billing activity, please share your invoice ID or registered account email through our secure support channel. Once we verify the account and billing details, we will provide an update on the next steps. Thank you for your patience.",
    "response_strategy": "Acknowledge concern, summarize issue, request verification details, and avoid making billing or refund commitments before review.",
    "assumptions": [
      "The billing issue has not yet been investigated.",
      "The account details have not yet been verified."
    ],
    "information_needed_before_sending": [
      "Invoice ID",
      "Registered account email"
    ]
  }
}
"""

draft_response_user_prompt = """
Customer Support Ticket:

{ticket_input}

Generate a customer response and return a JSON response that exactly follows the schema defined in the system prompt.
"""

draft_response_prompt = [
    {
        "role": "system",
        "content": draft_response_system_prompt
    },
    {
        "role": "user",
        "content": draft_response_user_prompt.format(
            ticket_input=ticket_input
        )
    }
]

# Model Call

response = call_groq_model(
    draft_response_prompt,
    temperature=0,
    max_tokens=600,
    
)

print(response)

{
  "draft_customer_response": {
    "draft_response": "Hi Amit, thank you for reaching out to us again. We apologize for the frustration caused by the duplicate charge and the lack of response to your previous inquiries. We understand your concern and would like to investigate this matter further. To assist us in reviewing your billing activity, could you please provide your invoice ID or the email address associated with your account? We will do our best to look into this and provide an update on the next steps.",
    "response_strategy": "Acknowledge concern, apologize for previous lack of response, request verification details to investigate the issue",
    "assumptions": [
      "The customer's account and billing details have not been verified",
      "The duplicate charge issue has not been previously investigated"
    ],
    "information_needed_before_sending": [
      "Invoice ID",
      "Registered account email"
    ]
  }
}
